# ENG-209, Partie 2, Série 8 (Cours)

## Introduction à la régression polynomiale avec Scikit-Learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from importlib import reload
from numpy.typing import NDArray

plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['font.size']      = 12

### Créons un jeu de données

On génère tout d'abord un ensemble de points aléatoires {$x_1^i$} pour $i: 1 \ldots N=20$

Ensuite, on génère les {$y^i$} tels que $y = x_1 + 2 x_1^2 + \epsilon$, où $\epsilon$ est un bruit gaussien de moyenne et écart type donnés: $\mu = 0$ et $\sigma = 4$.

In [ ]:
np.random.seed(0)

X: NDArray[np.float64] = np.sort(4 - 3 * np.random.normal(0, 1, 20))

y: NDArray[np.float64] = 5 + X + 2*(X**2) + np.random.normal(0, 4, 20)

X = X.reshape(-1, 1)  # Convertir en tableau 2D de forme (N, 1) pour l'entrée du modèle
y = y.reshape(-1,)    # Aplatir en tableau 1D de forme (N,) pour les valeurs cibles


### Voyons à quoi ressemblent les données

In [ ]:
reload(plt)

fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(X, y, color='#1f77b4')
ax.set_xlabel("$x_1$")
ax.set_ylabel("$y$")

plt.show()

### Modèle linéaire univarié

Et si on essayait d'approximer les {$y^i$} par un modèle linéaire univarié: $\hat{y} = w_0 + w_1 x_1$

In [ ]:
from sklearn.linear_model import LinearRegression

model_linear: LinearRegression = LinearRegression()
model_linear.fit(X, y)

y_pred_linear: NDArray[np.float64] = model_linear.predict(X)

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(X, y, color='#1f77b4', marker='o', label=r"$(x_1,y)$")
ax.plot(X, y_pred_linear, color='red', marker='x', label=r"$(x_1,\hat{y})$")

ax.set_xlabel("$x_1$")
ax.set_ylabel("$y$")
ax.legend(loc='best')
ax.set_title("Régression linéaire")

plt.show()

L’analyse des résidus a pour objectif de tester la validité d’un modèle de régression. Elle permet de déceler les défaillances d’un modèle, c’est pourquoi il est nécessaire de l’effectuer avant toute analyse de régression.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(X, y - y_pred_linear, color='#1f77b4')
ax.plot([-3,7], [0,0], color='red')

ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$y - \hat{y}$")
ax.set_title(r"Résiduels $y - \hat{y}$"+"\n"+"(régression linéaire)")

plt.show()

Si nous étudions le graphique ci-dessus, nous pouvons constater qu’une certaine strucure se dégage de la représentation des résidus.

Ce n'est clairement pas un bon modèle.

### Modèle polynomial

<strong>Idée</strong>: Dans notre exemple, on sait que la fonction $y = f(x_1)$ est de la forme $f(x) = w_0 + w_1 x + w_2 x_1^2$.

Et si nous créions une nouvelle variable indépendante $x_2 = x_1^2$ ? 

Nous pourrions dès lors approximer les $y^i$ par le plan $w_0 + w_1 x_1^i + w_2 x_2^i$ dont les paramètres $w^*$ s'obtiennent par la méthode des moindres carrées ordinaire (OLS).

Essayons

In [ ]:
X_transformed: NDArray[np.float64] = np.column_stack( (X, X**2) )

### Visualisons (en 3D) l'ensemble des points $( (x_1,x_2), y)$

In [ ]:
df: pd.DataFrame = pd.DataFrame(data=np.column_stack([X_transformed[:,0:2],y]), columns=['x1','x2','y'])

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(
    df['x1'],
    df['x2'],
    df['y'],
    s=16,
    c='#1f77b4',
    marker='o'
)

ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_zlabel('y')

# Camera position
ax.view_init(elev=0, azim=-90)

plt.show()

Les points semblent effectivement se trouver dans un plan

### Calculons les paramètres de ce plan

In [ ]:
model_poly: LinearRegression = LinearRegression()
model_poly.fit(X_transformed, y)

y_pred_poly: NDArray[np.float64] = model_poly.predict(X_transformed)

### Et visualisons le résultat

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Points réels
ax.scatter(X, y, marker='o', color='#1f77b4', label=r"$(x_1,y)$")

# Points et courbe prédictive calculée par la régression linéaire
ax.scatter(X, y_pred_poly, marker='x', color='red', label=r"$(x_1,\hat{y})$")
ax.plot(X, y_pred_poly, marker='x', color='red')

# Comparaison avec la fonction sous jacente à découvrir y = 5 + x + 2*x*x
ax.plot(X, 5+X+2*X*X, alpha=0.7, color='grey', linestyle=':', label=r"$y = x_1 + 2 x_1^2$")

ax.legend(loc='best')
ax.set_xlabel("$x_1$")
ax.set_ylabel("$y$")
ax.set_title("Régression polynomiale")


plt.show()

Analysons les résidus

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(X, y - y_pred_poly, marker='o', color='#1f77b4')
ax.plot([-3,7], [0,0], color='red')
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$y - \hat{y}$")
ax.set_title(r"Résiduels $y - \hat{y}$"+"\n"+"(régression polynomiale)")

plt.show()

Ce modèle semble adéquat. Voyons si l'hypothèse de normalité $(0, \sigma=4)$ de la distribution des résidus est vérifiée.

In [ ]:
from scipy.stats import norm

res: NDArray[np.float64] = y - y_pred_poly

mean_pdf, std_pdf = norm.fit(res)

x_pdf: NDArray[np.float64] = np.arange(-10, 10, 0.05, dtype=np.float64)

fig, ax = plt.subplots(figsize=(8, 6))

ax.hist(res, density=True, alpha=0.5, color='skyblue', edgecolor='black', label="Histogramme de densité")
ax.plot(x_pdf, norm.pdf(x_pdf, mean_pdf, std_pdf), color='red', label=fr"Loi normale ($\mu={mean_pdf:2.2f}$, $\sigma={std_pdf:2.2f}$)")

ax.legend(loc='best')
ax.set_xlabel(r"$y - \hat{y}$")
ax.set_ylabel("Histogramme de densité")
ax.set_title(r"Validation hypothèse de normalité des résidus $y - \hat{y}$")

plt.show()

Visualisons maintenant le plan de régression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create (x1,x2) meshgrid of [-5,10] x [-5,50]
A: NDArray[np.float64] = np.arange(-5, 10, 0.5, dtype=np.float64)
B: NDArray[np.float64] = np.arange(-5, 50, 0.5, dtype=np.float64)
A, B = np.meshgrid(A, B)

# Regression coefficients
w_star: NDArray[np.float64] = np.array([
    [model_poly.intercept_],
    [model_poly.coef_[0]],
    [model_poly.coef_[1]]
])

# Predicted values
Z: NDArray[np.float64] = (
    w_star[0] + w_star[1] * A + w_star[2] * B
)

# Create figure and 3D axes
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

# Fonction sous-jacente y = 5 + x1 + 2*x2
ax.plot(
    df['x1'],
    df['x2'],
    5 + df['x1'] + 2 * df['x2'],
    linestyle=':',
    linewidth=4,
    color='black'
)

# Points réels
ax.scatter(
    df['x1'],
    df['x2'],
    df['y'],
    s=16,
    color='#1f77b4'
)

# Plan prédictif calculé par la régression linéaire
ax.plot_surface(
    A, B, Z,
    alpha=0.6,
    cmap='viridis',
    edgecolor='none'
)

# Axis labels
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_zlabel('y')

# Figure dimensions
fig.set_size_inches(8, 8)

# Camera position
ax.view_init(elev=0, azim=-90)

plt.show()

### Voyons finalement comment la bibliothèque scikit-learn nous permet de simplifier ces différentes étapes

Introduisons des outils scikit-learn très utiles: **PolynomialFeatures** et **Pipeline**

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.base import TransformerMixin

polynomial_features: TransformerMixin = PolynomialFeatures(degree=2, include_bias=False)

X_transformed_pf: NDArray[np.float64] = polynomial_features.fit_transform( X )

Note : fit prépare, et fit_transform prépare puis applique la transformation. Pour la transformée polynomiale, cette étape de préparation (fit) ne fait rien.

In [ ]:
X_transformed_pf[:10,:]

**PolynomialFeatures** et **LinearRegression** peuvent être combinés dans un **Pipeline**.

la transformation polynomiale sert directement de variables d’entrée à la régression linéaire, et le pipeline obtenu se comporte comme un modèle et applique automatiquement les transformations.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# Pipeline combinant une transformation polynomiale et une régression linéaire.
# Nommer les étapes permet d’accéder aux paramètres de chaque étape via l’attribut .named_steps.
pipeline_linreg: Pipeline = Pipeline(
    steps=[
        ('polynomial', PolynomialFeatures(degree=2, include_bias=False)),  # création des termes polynomiaux
        ('regression', LinearRegression(fit_intercept=True)),              # régression linéaire
    ]
)

# Fit du pipeline sur X et y
# La transformation polynomiale est effectuée automatiquement avant la régression
pipeline_linreg.fit(X, y)

# Prédictions du pipeline.
y_pipeline_linreg_pred: NDArray[np.float64] = np.asarray(pipeline_linreg.predict(X))

In [ ]:
# Définir X_lisse et y_lisse_linreg pour tracer la courbe lissée prédite par le modèle sur l’intervalle [-3, 7].
X_lisse: NDArray[np.float64] = np.arange(-3, 7.0, 0.05, dtype=np.float64).reshape(-1,1)
y_lisse_linreg: NDArray[np.float64] = np.asarray(pipeline_linreg.predict(X_lisse))

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(X, y, marker='o', color='#1f77b4', label=r"$(x_1,y)$")
ax.scatter(X, y_pipeline_linreg_pred, marker='x', color='red', label=r"$(x_1,\hat{y})$")
ax.plot(X_lisse, y_lisse_linreg, color='red')
ax.plot(X_lisse, 5+X_lisse+2*X_lisse**2, alpha=0.7, color='grey', linestyle=':', label=r"$y=x_1+2x_1^2$")

ax.legend(loc='best')
ax.set_xlabel("$x_1$")
ax.set_ylabel("$y$")
ax.set_title("Régression polynomiale")

plt.show()

N'hésitez pas à essayer différentes "complexités" du polynômes (degré={1,2,5,10,15})

Qu'observez-vous?

### Régularisation pour éviter le surapprentissage

*️⃣ _Cette section fait appel à la notion d’optimisation itérative (descente), que nous verrons plus en détail plus tard._

Une méthode généralement utilisée est de pénaliser les valeurs extrêmes des paramètres $\bf{w}$, qui correspondent souvent à un _surapprentissage_, c'est à dire un modèle trop ajusté aux données d’apprentissage.

Pour cela, on ajoute à la fonction de coût que l'on cherche à minimiser un terme de pénalisation basé sur une norme des paramètres.

Les normes les plus fréquemment employées sont la norme L₁ (Lasso) et la norme L₂ (Ridge):

* La norme L₁ favorise une sélection de variables (certains coefficients deviennent exactement nuls), mais elle n’est pas _différentiable_, ce qui peut poser problème pour les méthodes d’optimisation itératives par gradient.

* La norme L₂ (que nous verrons également plus tard) est différentiable et conduit à une régularisation plus douce.

Considérons donc la fonction de coût suivante:

$w^* = \min_{w} { \frac{1}{2N} ||y - \tilde{X} w||_2 ^ 2 + \alpha ||w||_1}$

Où, par définition, $||w||_1 = \sum_{j=1}^m |w_j|$.

Utilisons maintenant la classe **Lasso** (L₁) de la bibliothèque scikit-learn, dont l'implémentation repose sur la descente de coordonnées pour ajusteer les coefficients.

**Attention**: les algorithmes d’optimisation itérative, comme celui utilisé ici, exigent généralement que les variables (dépendantes et indépendantes) soient standardisées.

**Standardisation**: Soit $x \sim \mathcal{N}(\mu,\,\sigma^2)$, alors $x_{\text{standardisé}} = \frac{x - \mu}{\sigma} \sim \mathcal{N}(0,\,1)$

Pour cela, nous utiliserons la classe _StandardScaler()_ de Scikit-Learn, que l’on peut intégrer directement dans un Pipeline.

In [ ]:
from sklearn.linear_model import Lasso

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

# Régression linéaire Lasso (i.e. avec régularisation)
pipeline_lasso: Pipeline = Pipeline(steps=[
    ('polynomial', PolynomialFeatures(degree=15, include_bias=False)),
    ('standardisation', StandardScaler()),
    ('regression', Lasso(alpha=0.1, fit_intercept=True)),
 ])

pipeline_lasso.fit(X, y)

y_pipeline_lasso_pred: NDArray[np.float64] = np.asarray(pipeline_lasso.predict(X))

In [ ]:
y_lisse_lasso_pred: NDArray[np.float64] = np.asarray(pipeline_lasso.predict(X_lisse))

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(X, y, marker='o', color='#1f77b4', label=r'$(x_1,y)$')
ax.scatter(X, y_pipeline_lasso_pred, color='red', marker='x', label=r'$(x_1,\hat{y})$')
ax.plot(X_lisse, y_lisse_lasso_pred, color='red')
ax.plot(X_lisse, 5+X_lisse+2*X_lisse*X_lisse, alpha=0.7, color='grey', linestyle=':', label=r'$y=x_1+2x_1^2$')

ax.legend(loc='best')
ax.set_xlabel("$x_1$")
ax.set_ylabel("$y$")
ax.set_title('Régression polynomiale régularisée par Lasso')

plt.show()

Pour obtenir les attributs d’un step dans un Pipeline, il suffit d’y accéder via la propriété named_steps['nom_du_step'].
Par exemple, pour récupérer les coefficients et l’interception (intercept) d’un step de régression linéaire, on peut procéder ainsi:

In [ ]:
pipeline_lasso.named_steps['regression'].coef_

In [ ]:
pipeline_lasso.named_steps['regression'].intercept_

In [ ]:
from sklearn.metrics import mean_absolute_error

print("Mean absolute error (training data): ", mean_absolute_error(y, y_pipeline_linreg_pred))
print("Mean absolute error (training data): ", mean_absolute_error(y, y_pipeline_lasso_pred))

---
⚠️ **Attention**:

Cet exemple, basé sur des données simulées, illustre simplement la **régression polynomiale** et l’usage des **pipelines**.

Nous n’avons pas séparé les données en **jeu d’entraînement** et **jeu de test**.

En pratique, cette étape est essentielle pour évaluer la performance réelle du modèle **et sa capacité de généralisation**.